# TD10: Ejective consonants and altitude

## 0. Question

Today, we will answer the following question:

*Are languages spoken at higher altitude more likely to have ejective consonants?*

We will use phonological inventory data from PHOIBLE and an elevation table based on the coordinates of each language. The goal is to replicate the results of [Everett (2013)](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0065275) and to understand the relationship between ejective consonants and altitude.


In [ ]:
# If Basemap is not installed, you can uncomment the following line. 
# IF YOU HAVE PROBLEMS INSTALLING BASEMAP, IGNORE THIS LINE.
# !pip install basemap

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

sns.set(context='notebook', style='ticks',
        font_scale=1.1, palette='colorblind')


## 1. Data

You will need to download `values.csv`, `parameters.csv` and `languages.csv` from [this Zenodo repository](https://zenodo.org/records/2677911). Then put it in the cldf forlder, which already contains the `elevation_data.csv` file.

In the `cldf` folder, you will find a CLDF version of PHOIBLE. We will use three files:

- `values.csv`: one row per segment in one language inventory
- `parameters.csv`: information about each segment, including phonological features
- `languages.csv`: language names, macroareas, and coordinates

The file `elevation_data.csv` was cached in advance from the [Open-Meteo Elevation API](https://open-meteo.com/en/docs/elevation-api), which estimates terrain elevation for latitude and longitude coordinates. 


In [ ]:
BASE = Path('.')
if not (BASE / 'cldf').exists():
    BASE = Path('S10')

values = pd.read_csv(BASE / 'cldf' / 'values.csv', low_memory=False)
parameters = pd.read_csv(BASE / 'cldf' / 'parameters.csv', low_memory=False)
languages = pd.read_csv(BASE / 'cldf' / 'languages.csv')
elevation = pd.read_csv(BASE /'cldf'/ 'elevation_data.csv')

values.shape, parameters.shape, languages.shape, elevation.shape

Let's inspect the segment lists. Each row corresponds to one segment (one phoneme, although they are encoded using IPA symbols) in one language inventory.


In [ ]:
values.head(10)

Now look at the language table. Which columns will we need for a geographical analysis?


In [ ]:
languages.head(10)


The parameter dataphrame is where we can find whether or not the segment is ejective. The relevant feature column is `raisedLarynxEjective`.


In [ ]:
parameters.columns


Let's extract the segment types that are marked as ejective. 


In [ ]:
ejective_segments = parameters.query('raisedLarynxEjective == "+"')
ejective_segments[['Name', 'Description', 'SegmentClass', 'raisedLarynxEjective']].head(20)

How many distinct segment types in PHOIBLE are marked as ejective?


In [ ]:
##################
# YOUR CODE HERE #
##################

Now combine the segment observations with the feature table. This will let us know, for every segment observed in every language, whether that segment is an ejective consonant. Keep only the following columns in the parameters dataframe: `ID`,`Name`, `SegmentClass`, `raisedLarynxEjective`. Call the resulting dataframe `segments`.


In [ ]:
##################
# YOUR CODE HERE #
##################

segments.head(10)


PHOIBLE can contain more than one inventory contribution for the same language. Today, we will count each segment type only once per language. The code below removes duplicate inventory contributions for the same language, keeping only the first one.


In [ ]:
# Keep only one inventory contribution per language (choose the smallest Contribution_ID)
one_contrib_per_lang = segments.groupby('Language_ID')['Contribution_ID'].min()

segments_unique = segments[
    segments['Contribution_ID'].eq(segments['Language_ID'].map(one_contrib_per_lang))
].copy()

# Safety: keep each segment type only once per language
segments_unique = segments_unique.drop_duplicates(['Language_ID', 'Parameter_ID'])

segments.shape, segments_unique.shape

Create the `is_ejective` column, which is `True` if the segment is an ejective consonant and `False` otherwise. Also create the `is_consonant` column, which is `True` if the segment is a consonant and `False` otherwise.

In [ ]:
##################
# YOUR CODE HERE #
##################

segments_unique[['Language_ID', 'Value', 'Name', 'SegmentClass', 'is_ejective']].head(10)


Now we can move from segment-level data to language-level data. For each language, count the total number of distinct segments, the number of consonants, and the number of ejectives. Use the `groupby` method to group by `Language_ID`, and then use the `agg` method to count the number of segments, consonants, and ejectives. Call the resulting dataframe `lang_ejectives`.


In [ ]:
##################
# YOUR CODE HERE #
##################


Add a binary column `has_ejective` that is `True` if the language has at least one ejective consonant and `False` otherwise. Also add a column `has_ejective_int` that is `1` if the language has at least one ejective consonant and `0` otherwise. We will use this column for the correlation analysis later on.

In [ ]:
##################
# YOUR CODE HERE #
##################

Which languages have the largest number of ejective segment types in this dataset?


In [ ]:
lang_ejectives.sort_values('n_ejectives', ascending=False).head(15)

Do you find this number to be plausible? Why or why not? Can you find out more about this language?

Next, add language names, macroareas, coordinates, and elevation. You would need to merge the `lang_ejectives` dataframe with the `languages_small` and `elevation_data` dataframes. The `Language_ID` column is the key for merging with the `languages` dataframe, and the `Language_ID` column is also the key for merging with the `elevation_data` dataframe. Save this merged dataframe as `f`.


In [ ]:
languages_small = languages[['ID', 'Name', 'Macroarea', 'Latitude', 'Longitude', 'Family_Name']]

##################
# YOUR CODE HERE #
##################


Now drop rows without an elevation estimate, and convert them to kilometers by dividing by 1000. How many languages do we have in the final dataset?

In [ ]:
f['elevation_m'] = pd.to_numeric(f['elevation_m'], errors='coerce')
print('Missing elevations:', f['elevation_m'].isna().sum())

##################
# YOUR CODE HERE #
##################

Let's look at the results:

In [ ]:
df.head(10)


Before modeling anything, check the outcome variable. What proportion of languages in this dataset have at least one ejective consonant?


In [ ]:
##################
# YOUR CODE HERE #
##################

## 2. Exploratory analysis

Let's first look at the elevation variable itself. Do the minimum and maximum values look plausible?


In [ ]:
df['elevation_m'].describe()


Which languages in the sample have the highest estimated elevation? Do they all have ejectives?


In [ ]:
##################
# YOUR CODE HERE #
##################

Now compare the elevation distributions for languages with and without ejectives.


In [ ]:
##################
# YOUR CODE HERE #
##################

Perform a two-sample t-test. Are languages with ejectives spoken at higher estimated elevations than languages without ejectives?


In [ ]:
##################
# YOUR CODE HERE #
##################

## 3. Logistic regression

A simple way to model a binary outcome is logistic regression. Here, the outcome is whether a language has at least one ejective consonant. The predictor is elevation in kilometers. Use the `smf.logit` function from the statsmodels library to fit a logistic regression model. What is the effect of elevation on the probability of having ejectives? Is it significant?


In [ ]:
##################
# YOUR CODE HERE #
##################

Convert the coefficient into an odds ratio. This tells us how the odds of having an ejective change for each additional kilometer of elevation.


In [ ]:
odds_ratio = np.exp(model.params['elevation_km'])
print(f'Estimated odds ratio for a 1 km increase: {odds_ratio:.2f}')


Plot the results of the regression using the `regplot` function from seaborn. You can read about it [here](https://seaborn.pydata.org/generated/seaborn.regplot.html). 

In [ ]:
plt.figure(figsize=(8, 4))
sns.regplot(
    data=df,
    x='elevation_m',
    y='has_ejective_int',
    logistic=True,
    y_jitter=0.03,
    scatter_kws={'alpha': 0.25, 's': 18},
    line_kws={'color': 'black'}
)
plt.xlabel('Elevation, meters')
plt.ylabel('Probability of at least one ejective')
sns.despine()
plt.show()


## 4. A first look at confounding

Languages are not independent points scattered randomly around the globe. Before trusting the altitude effect too much, let's check whether ejectives are concentrated in particular macroareas.


In [ ]:
macroarea_summary = (
    df.groupby('Macroarea')['has_ejective']
    .agg(probability='mean', n_ejectives='sum', n_languages='count')
    .sort_values('probability', ascending=False)
)

macroarea_summary


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=macroarea_summary.reset_index(),
    x='Macroarea',
    y='probability',
    color='steelblue'
)
plt.xlabel('Macroarea')
plt.ylabel('Probability of at least one ejective')
plt.xticks(rotation=30, ha='right')
sns.despine()
plt.show()


As an optional extension, fit a model that includes macroarea. In this sample, Australia and Papunesia have no or almost no languages with ejectives, which makes the logistic model unstable, so we will remove them from our sample. 

Run the new model, adding another categorical predictor for macroarea using the `C()` function from statsmodels. How does the altitude effect change when we control for macroarea?

In [ ]:
##################
# YOUR CODE HERE #
##################

What can you tell about the relationship between ejectives and altitude after controlling for macroarea? Does this change your interpretation of the results?

Finally, make a map. Are the ejective languages spread evenly across the world, or do they cluster  geographically? You can take inspiration from the map we made in [TD8](https://github.com/alexeykosh/intro-to-ling-2026/blob/main/s8/TD8_clics_database_solutions.ipynb).


In [ ]:
##################
# YOUR CODE HERE #
##################